# COMP3122 — Assignment 2

**Applied Data Science · 12% of the course · 100 marks · due Sunday, November 29, 11:59 p.m. (America/Toronto)**

## The situation

You are a data-science consultant again. This time the client is the fleet
maintenance manager of a city bus service. She has sent the memo below and an
export of every brake inspection her team logged last year, one row per
inspection.

This notebook is your answer. It has seven parts:

- Part A: the fleet, and the picture
- Part B: how fast pads wear (linear regression)
- Part C: two classifiers (KNN and logistic regression)
- Part D: which model costs least
- Part E: moving the threshold, and what it costs
- Part F: what you would tell her
- the AI Usage Declaration

Read the memo, then read the whole notebook once before you begin.

---

> **MEMORANDUM**
>
> **To:** Data Science consultant
>
> **From:** Fleet Maintenance Manager
>
> **Re:** Replacing fixed brake checks with a model
>
> Today every bus comes in for a brake check on a fixed schedule. Most of
> those checks find pads with plenty left; a few buses wear through before
> their turn comes. A pad under **3.0 mm** is below our safety limit.
>
> **I want to know whether a model can tell me which buses to pull in early,
> and how much it would save us.**
>
> Attached is last year's inspection export: kilometres since the last brake
> service, how hilly the route is, stops per km, average passenger load, the
> driver's braking score, the bus's age, and the pad thickness we measured.
>
> Two things matter to me. A bus that should have come in and didn't is the
> expensive mistake. And my team swears old buses eat pads — I want that
> checked before anyone builds it into a schedule.

---

## How this is marked

| | Marks | Marked by |
|---|---|---|
| Part A — the fleet, and the chart | **15** | Automatically; the chart from its Axes object |
| Part B — linear regression | **15** | Automatically |
| Part C — KNN and logistic regression | **15** | Automatically |
| Part D — which model costs least | **15** | Automatically |
| Part E — the threshold and the trade | **20** | Automatically |
| Part F — your conclusions | **10** | Automatically |
| AI Usage Declaration — complete and signed | **10** | Automatically |
| | **100** | |

**Your answers are marked, and they must come from your code.** Each part
lists the values to report and the precision required. Where a part names a
model or a split, use exactly that one, so that everyone's experiment is the
same; beyond that, the method is yours. Parts B and C ask you to store your
**fitted models**, and Part E your **probabilities**. Tests check the specified
experiment and that your stored objects agree with your reported values. You
remain responsible for producing the work yourself.

**You may run `Validate` as often as you wish** before you submit. It runs the
same tests that are used to mark your submission; there are no hidden tests. If
Validate reports no errors, your submission will pass every test.

**Parts E and F carry the judgement.** Correct numbers with the wrong
conclusion lose marks.

## Rules

- **Don't copy and paste cells in the assignment notebook.** To try something, add a new cell with **+**. Never change a cell's type or the grey cells. If you pasted a copy of a cell, delete the copy before you submit.
- Complete the parts in order. Later parts use results from earlier ones.
- **Do not reassign `inspections`.** Store any filtered data under a new name.
- Use the exact strings shown wherever a value is a choice.
- This assignment uses Chapters 5–8 only. Nothing needs `groupby`, cleaning, or cross-validation.
- AI use is permitted and must be **declared in the last section of this notebook**. Nothing is submitted on D2L.
- Submissions are compared with each other automatically. Similar code is reviewed under the academic-integrity policy.
- Late work: 20 marks out of 100 are deducted for each started day late, to a minimum of 0; work five or more days late receives 0.


## Setup — run this cell first

This cell loads the export into `inspections`, names the six feature columns,
sets the two costs and the thresholds to try, and creates an empty dictionary,
`results`, where you store the value each part asks for. Do not change this cell.

In [ ]:
# Run this first. It loads the export and creates the dict you report into.
import numpy as np
import pandas as pd

inspections = pd.read_csv("data/inspections.csv")

# The six numeric columns every model in this assignment uses, in this order.
FEATURES = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']

COST_MISS = 4000     # dollars: a bus below the limit that was not pulled in
COST_PULL = 250      # dollars: a bus pulled in that was not below the limit
THRESHOLDS = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]

# Every autograded value goes in here, under the key the part names.
results = {}

print("loaded", inspections.shape[0], "inspections x", inspections.shape[1], "columns")
inspections.head()

### Part A — The fleet, and the picture · 15 marks

Before any model, describe what the manager has given you, and show her the
pattern the models will learn from.

A bus is **below the limit** when `pad_mm` is under **3.0**. That is the bus the
manager most needs to catch.

**Store the following values in `results`:**

| key | description |
|---|---|
| `n_inspections` | how many inspections the export covers |
| `share_below_limit` | the share of all inspections that were below the limit, 3 decimals |
| `depot_most_below_limit` | the depot with the **highest share** of its inspections below the limit, as a string exactly as it appears in the data |
| `ax_wear` | the Axes of the chart described below |

**The chart.** A scatter plot of every inspection: `km_since_service` on the
x-axis, `pad_mm` on the y-axis, one point per inspection. Draw the **3.0 mm
safety limit** as a horizontal line across it. Give it a title, an x-axis label
that names the distance in km, and a y-axis label that names the pad thickness.
The tests read the Axes object (points, labels, the limit line), not the image.

*For reference: `fig, ax = plt.subplots()` and `ax.scatter(...)` are Chapter 5;
a horizontal line across an Axes is `ax.axhline(3.0)`, as in Chapter 6.*

In [ ]:
# YOUR WORKING HERE
pass


In [ ]:
assert "n_inspections" in results, "results['n_inspections'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("n_inspections", _canon("n_inspections", results["n_inspections"], "int")) in {'b473bae06f1697cc'}, \
    "results['n_inspections'] is not correct yet"
print("ok")

In [ ]:
assert "share_below_limit" in results, "results['share_below_limit'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("share_below_limit", _canon("share_below_limit", results["share_below_limit"], "float")) in {'bca4e771668499da', 'df418b66aa14d4bb', 'cf55481aa58166b1'}, \
    "results['share_below_limit'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "depot_most_below_limit" in results, "results['depot_most_below_limit'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("depot_most_below_limit", _canon("depot_most_below_limit", results["depot_most_below_limit"], "str")) in {'527be8694730cfa3'}, \
    "results['depot_most_below_limit'] is not correct yet — use the exact string shown in the brief"
print("ok")

In [ ]:
assert "ax_wear" in results, "results['ax_wear'] is not set — store the Axes you drew on"
import numpy as np
import pandas as pd
import matplotlib.axes
from matplotlib.collections import PathCollection, LineCollection
ax = results["ax_wear"]
assert isinstance(ax, matplotlib.axes.Axes), "results['ax_wear'] must be a Matplotlib Axes"
_d = pd.read_csv("data/inspections.csv")
def _cyc(a, n):
    a = np.asarray(a, dtype=float)
    if a.ndim == 0:
        a = a.reshape(1)
    if len(a) == 0:
        return np.zeros(n)
    return a[np.arange(n) % len(a)]
def _shown(c, n, use_size):
    c.update_scalarmappable()
    fc = np.asarray(c.get_facecolor(), dtype=float).reshape(-1, 4)
    ec = np.asarray(c.get_edgecolor(), dtype=float).reshape(-1, 4)
    lw = _cyc(c.get_linewidth(), n)
    face = _cyc(fc[:, 3], n) if len(fc) else np.zeros(n)
    edge = (_cyc(ec[:, 3], n) if len(ec) else np.zeros(n)) * (lw > 0)
    ok = (face > 0) | (edge > 0)
    if use_size:
        ok &= _cyc(c.get_sizes(), n) > 0
    return ok
ax.get_xlim(); ax.get_ylim()                     # explain: settle the axis limits first
_pts = []
for _c in ax.collections:
    if not isinstance(_c, PathCollection) or not _c.get_visible():
        continue
    _o = np.ma.filled(np.ma.asarray(_c.get_offsets(), dtype=float), np.nan).reshape(-1, 2)
    _keep = _shown(_c, len(_o), use_size=True)
    _pts.append(_c.get_offset_transform().transform(_o)[_keep])
_pts = np.vstack(_pts) if _pts else np.empty((0, 2))
_pts = _pts[np.all(np.isfinite(_pts), axis=1)]
assert len(_pts) == 1200, f"expected one visible scatter point per inspection, found {len(_pts)} — check the data and that the points are visible"
_want = ax.transData.transform(_d[["km_since_service", "pad_mm"]].to_numpy(dtype=float))
_a = _pts[np.lexsort((_pts[:, 1], _pts[:, 0]))]
_b = _want[np.lexsort((_want[:, 1], _want[:, 0]))]
assert np.allclose(_a, _b, atol=0.05), \
    "the scatter points are not km_since_service (x) against pad_mm (y), one per inspection, on the chart"
print("ok")

In [ ]:
assert "ax_wear" in results, "results['ax_wear'] is not set — store the Axes you drew on"
import numpy as np
import pandas as pd
import matplotlib.axes
from matplotlib.collections import PathCollection, LineCollection
ax = results["ax_wear"]
assert isinstance(ax, matplotlib.axes.Axes), "results['ax_wear'] must be a Matplotlib Axes"
_d = pd.read_csv("data/inspections.csv")
def check_labelled(label, ax):
    """The four-line minimum, checked on a real Axes."""
    if callable(ax) and not hasattr(ax, "get_title"):
        try:
            ax = ax()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"        reaching that Axes raised {type(e).__name__}: {e}")
            return
    if not hasattr(ax, "get_title"):
        print(f"NOT YET  {label}")
        print(f"        that is a {type(ax).__name__}, not an Axes — did you store `fig`"
              f" where you meant `ax`?")
        return
    parts = (("title", ax.get_title()), ("x-label", ax.get_xlabel()),
             ("y-label", ax.get_ylabel()))
    missing = [name for name, value in parts if not value.strip()]
    if missing:
        print(f"NOT YET  {label}")
        print(f"        missing: {', '.join(missing)}")
        print(f"        you have: title={ax.get_title()!r}, x={ax.get_xlabel()!r},"
              f" y={ax.get_ylabel()!r}")
    else:
        print(f"PASS  {label}")
import io, contextlib
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    check_labelled("Part A chart", ax)
assert _buf.getvalue().startswith("PASS"), _buf.getvalue().strip()
assert "km" in ax.get_xlabel().lower(), f"x-axis label should name the distance in km, got {ax.get_xlabel()!r}"
assert "mm" in ax.get_ylabel().lower() or "pad" in ax.get_ylabel().lower(), \
    f"y-axis label should name the pad thickness (mm), got {ax.get_ylabel()!r}"
print("ok")

In [ ]:
assert "ax_wear" in results, "results['ax_wear'] is not set — store the Axes you drew on"
import numpy as np
import pandas as pd
import matplotlib.axes
from matplotlib.collections import PathCollection, LineCollection
ax = results["ax_wear"]
assert isinstance(ax, matplotlib.axes.Axes), "results['ax_wear'] must be a Matplotlib Axes"
_d = pd.read_csv("data/inspections.csv")
def _cyc(a, n):
    a = np.asarray(a, dtype=float)
    if a.ndim == 0:
        a = a.reshape(1)
    if len(a) == 0:
        return np.zeros(n)
    return a[np.arange(n) % len(a)]
def _shown(c, n, use_size):
    c.update_scalarmappable()
    fc = np.asarray(c.get_facecolor(), dtype=float).reshape(-1, 4)
    ec = np.asarray(c.get_edgecolor(), dtype=float).reshape(-1, 4)
    lw = _cyc(c.get_linewidth(), n)
    face = _cyc(fc[:, 3], n) if len(fc) else np.zeros(n)
    edge = (_cyc(ec[:, 3], n) if len(ec) else np.zeros(n)) * (lw > 0)
    ok = (face > 0) | (edge > 0)
    if use_size:
        ok &= _cyc(c.get_sizes(), n) > 0
    return ok
from matplotlib.colors import to_rgba
ax.get_xlim(); ax.get_ylim()                     # explain: settle the axis limits first
_lo, _hi = float(_d["km_since_service"].min()), float(_d["km_since_service"].max())
_tgt = ax.transData.transform([[_lo, 3.0], [_hi, 3.0]])
_xlo, _xhi = sorted(_tgt[:, 0])
_ty = _tgt[0, 1]
def _runs(xy):
    # explain: the unbroken pieces of a stroke: consecutive finite vertices only
    ok = np.all(np.isfinite(xy), axis=1)
    out, cur = [], []
    for _p, _k in zip(xy, ok):
        if _k:
            cur.append(_p)
        else:
            out.append(cur); cur = []
    out.append(cur)
    return [np.asarray(r) for r in out if len(r) >= 2]
def _span(xy):
    for r in _runs(xy):
        if np.all(np.abs(r[:, 1] - _ty) <= 1.0):
            yield r[:, 0].min(), r[:, 0].max()
_spans = []
for _l in ax.lines:
    if not _l.get_visible() or str(_l.get_linestyle()).lower() in ("none", "", " "):
        continue
    # explain: as Line2D.draw does it — an explicit alpha REPLACES the colour's own alpha (round 4 #4)
    _al = to_rgba(_l.get_color(), _l.get_alpha())[3]
    if not (_l.get_linewidth() > 0 and _al > 0):
        continue
    _x = np.ma.filled(np.ma.asarray(_l.get_xdata(), dtype=float), np.nan)
    _y = np.ma.filled(np.ma.asarray(_l.get_ydata(), dtype=float), np.nan)
    _spans += list(_span(_l.get_transform().transform(np.column_stack([_x, _y]))))
for _c in ax.collections:
    if not isinstance(_c, LineCollection) or not _c.get_visible():
        continue
    # explain: the drawn paths, with their NaN/masked vertices and sub-path starts kept as breaks
    # explain: (get_segments() drops them); a stroke needs a visible EDGE and a positive width —
    # explain: a face colour draws nothing on a line (round 4 #4)
    _segs = []
    for _pth in _c.get_paths():
        _v = np.ma.filled(np.ma.asarray(_pth.vertices, dtype=float), np.nan).reshape(-1, 2)
        if _pth.codes is not None:
            _brk = [i for i, k in enumerate(_pth.codes) if k == 1 and i > 0]   # explain: 1 = MOVETO
            _v = np.insert(_v, _brk, np.nan, axis=0)
        _segs.append(_v)
    _c.update_scalarmappable()
    _ec = np.asarray(_c.get_edgecolor(), dtype=float).reshape(-1, 4)
    _off = np.asarray(_c.get_offsets(), dtype=float).reshape(-1, 2)
    # explain: Matplotlib draws max(paths, offsets) strokes, cycling the shorter list,
    # explain: and cycles widths and colours over the same count (round 5 #1)
    _n = max(len(_segs), len(_off)) if _segs else 0
    _keep = (_cyc(_c.get_linewidth(), _n) > 0) & ((_cyc(_ec[:, 3], _n) if len(_ec) else np.zeros(_n)) > 0)
    _ot = _c.get_offset_transform()
    for _i in range(_n):
        _s = _segs[_i % len(_segs)]
        if not _keep[_i]:
            continue
        _xy = _c.get_transform().transform(_s)
        if len(_off):
            _xy = _xy + _ot.transform(_off[_i % len(_off)].reshape(1, 2))  # explain: where Matplotlib really draws it
        _spans += list(_span(_xy))
assert any(a <= _xlo + 1 and b >= _xhi - 1 for a, b in _spans), \
    "draw the 3.0 mm safety limit as a visible horizontal line across the whole range of the data"
print("ok")

### Part B — How fast do pads wear? · 15 marks

> *"My team says old buses eat brake pads. Before you tell me which buses to
> pull in, tell me what actually wears a pad down."*

Fit a **linear regression** that predicts `pad_mm` from the six numeric columns
in `FEATURES`. Use **exactly** this split, so your numbers can be compared with
everyone else's:

```
train_test_split(inspections[FEATURES], inspections["pad_mm"], test_size=0.2, random_state=42)
```

**Store the following values in `results`:**

| key | description |
|---|---|
| `lin_model` | your fitted `LinearRegression`, trained on the training rows only |
| `test_r2` | R² of your model on the **test** rows, 3 decimals |
| `rmse_mm` | how far off a typical test prediction is, in mm: the square root of the test mean squared error, 3 decimals |
| `mm_per_10000km` | how many mm of pad a bus loses per **10,000 km**, all else equal — a positive number, 3 decimals |
| `mm_per_year_of_age` | how much `pad_mm` changes per extra year of bus age, all else equal, 3 decimals (keep the sign) |
| `age_drives_wear` | `True` or `False`: does your model support the team's belief that old buses wear pads faster? Compare how much pad a bus's whole age range (1 to 15 years) moves the prediction with how much the other features move it. |

The tests check `lin_model` itself as well as the numbers: they rebuild the test
rows and ask your model. Keep the stored model the one you reported from — don't
refit it afterwards.

In [ ]:
# YOUR WORKING HERE
pass


In [ ]:
assert "lin_model" in results, "results['lin_model'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_m = results["lin_model"]
assert type(_m).__name__ == "LinearRegression", f"results['lin_model'] must be a fitted LinearRegression, got {type(_m).__name__}"
assert hasattr(_m, "coef_") and len(np.ravel(_m.coef_)) == 6, "lin_model is not fitted on the six FEATURES yet"
_cols(_m)
_ok = _fp("lin_r2", _canon("lin_r2", _r2(), "float")) in {'fc084dd3f9547ef8', '4c8c63cf1c8992b0', '05da7dfe411854de'}
_ok = _ok and _fp("lin_b", format(round(_b(_m), 2), ".2f")) == 'cebf2dfafc12056c'
assert _ok, "lin_model does not reproduce the expected model — fit it on the training rows of exactly the split in the brief, with the six FEATURES"
print("ok")

In [ ]:
assert "test_r2" in results, "results['test_r2'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('lin_model')
_v = _r2()
assert abs(_num("test_r2", results["test_r2"]) - float(_v)) <= 0.0006, \
    "results['test_r2'] is not what your stored objects give — report the value your code computes"
assert _fp("test_r2", _canon("test_r2", results["test_r2"], "float")) in {'b36df066345d2ad1', '675b8df02b38c52e', '7cfb8d2f583e84c1'}, \
    "results['test_r2'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "rmse_mm" in results, "results['rmse_mm'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('lin_model')
_v = float(np.sqrt(np.mean((_lin() - _mte.values) ** 2)))
assert abs(_num("rmse_mm", results["rmse_mm"]) - float(_v)) <= 0.0006, \
    "results['rmse_mm'] is not what your stored objects give — report the value your code computes"
assert _fp("rmse_mm", _canon("rmse_mm", results["rmse_mm"], "float")) in {'e8cade2e66665267', 'e3fa5fafd7aa705e', 'e77501df899bb133'}, \
    "results['rmse_mm'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "mm_per_10000km" in results, "results['mm_per_10000km'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('lin_model')
_m = _model("lin_model")
_v = -dict(zip(_cols(_m), np.ravel(_m.coef_)))["km_since_service"] * 10000
assert abs(_num("mm_per_10000km", results["mm_per_10000km"]) - float(_v)) <= 0.0006, \
    "results['mm_per_10000km'] is not what your stored objects give — report the value your code computes"
assert _fp("mm_per_10000km", _canon("mm_per_10000km", results["mm_per_10000km"], "float")) in {'e21b68d14d1253d4', '16cb6a52779247ae', 'a20b2f8a541235db'}, \
    "results['mm_per_10000km'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "mm_per_year_of_age" in results, "results['mm_per_year_of_age'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('lin_model')
_m = _model("lin_model")
_v = dict(zip(_cols(_m), np.ravel(_m.coef_)))["bus_age_years"]
assert abs(_num("mm_per_year_of_age", results["mm_per_year_of_age"]) - float(_v)) <= 0.0006, \
    "results['mm_per_year_of_age'] is not what your stored objects give — report the value your code computes"
assert _fp("mm_per_year_of_age", _canon("mm_per_year_of_age", results["mm_per_year_of_age"], "float")) in {'d2ade3fb6a4ab677', '8e95524e5414d625', '7638fa9f7e53c1c6'}, \
    "results['mm_per_year_of_age'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "age_drives_wear" in results, "results['age_drives_wear'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("age_drives_wear", _canon("age_drives_wear", results["age_drives_wear"], "bool")) in {'2a79ddbb283c2952'}, \
    "results['age_drives_wear'] is not correct yet"
print("ok")

### Part C — Two classifiers, and the number to beat · 15 marks

Now treat the same question as **classification**: is a bus below the limit or
not? Use the **same split** as Part B, and label each bus `1` if its `pad_mm` is
under 3.0 and `0` otherwise. `1` is the positive class: the bus you are trying to
catch.

Fit two classifiers on the six `FEATURES`, both on **scaled** features, with the
scaler fitted on the training rows only (Chapter 7):

- `KNeighborsClassifier(n_neighbors=5)`
- `LogisticRegression()` with its default settings (you may raise `max_iter` if it
  warns that it did not converge; nothing else)

**Store the following values in `results`:**

| key | description |
|---|---|
| `scaler` | your fitted `StandardScaler` |
| `knn_model` | your fitted `KNeighborsClassifier` |
| `log_model` | your fitted `LogisticRegression` |
| `baseline_accuracy` | test accuracy of a "model" that says *every bus is fine*, 3 decimals |
| `knn_accuracy` | KNN's test accuracy, 3 decimals |
| `knn_recall` | KNN's test recall for buses below the limit, 3 decimals |
| `log_accuracy` | logistic regression's test accuracy, 3 decimals |
| `log_precision` | logistic regression's test precision for buses below the limit, 3 decimals |
| `log_recall` | logistic regression's test recall for buses below the limit, 3 decimals |

Compare both accuracies with `baseline_accuracy` before you trust either.

*Column order:* if you fit on DataFrames, the tests use your column names. If you
fit on plain arrays (for example the output of `scaler.transform`), the columns
must be in `FEATURES` order, because an array carries no names.

In [ ]:
# YOUR WORKING HERE
pass


In [ ]:
assert "scaler" in results, "results['scaler'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_m = results["scaler"]
assert type(_m).__name__ == "StandardScaler", f"results['scaler'] must be a fitted StandardScaler, got {type(_m).__name__}"
assert hasattr(_m, "mean_") and hasattr(_m, "scale_"), "scaler is not fitted yet"
assert _m.with_mean and _m.with_std, "scaler must centre AND scale (with_mean=True, with_std=True)"
_c = _cols(_m)
assert not (np.allclose(_m.mean_, _d[_c].mean().values) and not np.allclose(_m.mean_, _Xtr[_c].mean().values)), \
    "scaler was fitted on all the rows — fit it on the training rows only"
assert np.allclose(_m.mean_, _Xtr[_c].mean().values) and np.allclose(_m.scale_, _Xtr[_c].std(ddof=0).values), \
    "scaler was not fitted on the training rows of the split in the brief"
print("ok")

In [ ]:
assert "knn_model" in results, "results['knn_model'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_m = results["knn_model"]
assert type(_m).__name__ == "KNeighborsClassifier", f"results['knn_model'] must be a fitted KNeighborsClassifier, got {type(_m).__name__}"
assert hasattr(_m, "classes_"), "knn_model is not fitted yet"
assert _m.n_neighbors == 5, f"knn_model must use n_neighbors=5, has {_m.n_neighbors}"
# explain: the settings the brief fixes; algorithm and leaf_size only change speed, so they are free
assert _m.weights == "uniform", "knn_model must use the default weights='uniform'"
assert (_m.metric == "minkowski" and _m.p == 2) or _m.metric == "euclidean", \
    "knn_model must use the default distance (Euclidean)"
_need("scaler")
_p = _cls("knn_model")
assert _fp("knn_pred", "".join(map(str, _p))) == '99563521e4a947e2', \
    "knn_model does not reproduce the expected test predictions — scaled features, training rows only, n_neighbors=5"
print("ok")

In [ ]:
assert "log_model" in results, "results['log_model'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_m = results["log_model"]
assert type(_m).__name__ == "LogisticRegression", f"results['log_model'] must be a fitted LogisticRegression, got {type(_m).__name__}"
assert hasattr(_m, "coef_"), "log_model is not fitted yet"
# explain: LogisticRegression() with its defaults; only max_iter (how long it may search) is free
_want = {"C": 1.0, "penalty": "l2", "solver": "lbfgs", "class_weight": None, "fit_intercept": True,
         "intercept_scaling": 1, "l1_ratio": None, "dual": False, "tol": 0.0001}
_got = _m.get_params()
_off = sorted(k for k, v in _want.items() if _got.get(k) != v)
assert not _off, f"log_model must use LogisticRegression() with its default settings — changed: {_off}"
_need("scaler")
_p = _cls("log_model")
assert _fp("log_pred", "".join(map(str, _p))) == 'dbea6aa0e2a36e1b', \
    "log_model does not reproduce the expected test predictions — scaled features, training rows only, default settings"
print("ok")

In [ ]:
assert "baseline_accuracy" in results, "results['baseline_accuracy'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("baseline_accuracy", _canon("baseline_accuracy", results["baseline_accuracy"], "float")) in {'9d658bb709a99e0f', '9aae3299d372ce69', '8fb8dc1057c002a1'}, \
    "results['baseline_accuracy'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "knn_accuracy" in results, "results['knn_accuracy'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('knn_model')
_p = _cls("knn_model")
_v = float((_p == _yte).mean())
assert abs(_num("knn_accuracy", results["knn_accuracy"]) - float(_v)) <= 0.0006, \
    "results['knn_accuracy'] is not what your stored objects give — report the value your code computes"
assert _fp("knn_accuracy", _canon("knn_accuracy", results["knn_accuracy"], "float")) in {'9e8775316af85e3d', 'c784eb7d2e0de9cc', '6bb944e9fd5750ad'}, \
    "results['knn_accuracy'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "knn_recall" in results, "results['knn_recall'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('knn_model')
_p = _cls("knn_model")
_v = float(((_p == 1) & (_yte == 1)).sum() / (_yte == 1).sum())
assert abs(_num("knn_recall", results["knn_recall"]) - float(_v)) <= 0.0006, \
    "results['knn_recall'] is not what your stored objects give — report the value your code computes"
assert _fp("knn_recall", _canon("knn_recall", results["knn_recall"], "float")) in {'1937adc220516cd7', '3e85892c26fdb013', 'b3ab9ba37156b9db'}, \
    "results['knn_recall'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "log_accuracy" in results, "results['log_accuracy'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('log_model')
_p = _cls("log_model")
_v = float((_p == _yte).mean())
assert abs(_num("log_accuracy", results["log_accuracy"]) - float(_v)) <= 0.0006, \
    "results['log_accuracy'] is not what your stored objects give — report the value your code computes"
assert _fp("log_accuracy", _canon("log_accuracy", results["log_accuracy"], "float")) in {'63ace89a4eafb391', 'd6a2d16be0f3988e', '1140eeedc5400d87'}, \
    "results['log_accuracy'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "log_precision" in results, "results['log_precision'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('log_model')
_p = _cls("log_model")
_v = float(((_p == 1) & (_yte == 1)).sum() / max(1, (_p == 1).sum()))
assert abs(_num("log_precision", results["log_precision"]) - float(_v)) <= 0.0006, \
    "results['log_precision'] is not what your stored objects give — report the value your code computes"
assert _fp("log_precision", _canon("log_precision", results["log_precision"], "float")) in {'5cae6f3b302e9c28', '37ffaeed7c911640', '6caefa1d82aaba6a'}, \
    "results['log_precision'] is not correct yet (to 3 decimals)"
print("ok")

In [ ]:
assert "log_recall" in results, "results['log_recall'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('log_model')
_p = _cls("log_model")
_v = float(((_p == 1) & (_yte == 1)).sum() / (_yte == 1).sum())
assert abs(_num("log_recall", results["log_recall"]) - float(_v)) <= 0.0006, \
    "results['log_recall'] is not what your stored objects give — report the value your code computes"
assert _fp("log_recall", _canon("log_recall", results["log_recall"], "float")) in {'5428f9eecebf7f07', '5f75d1193bd014cc', '300813031bdd87fa'}, \
    "results['log_recall'] is not correct yet (to 3 decimals)"
print("ok")

### Part D — Which model would you believe? · 15 marks

You now have three ways to flag a bus:

- **linear regression**: flag a bus when its *predicted* `pad_mm` is under 3.0
- **KNN**: its `.predict()`
- **logistic regression**: its `.predict()` (threshold 0.5)

> *"I don't care which one is most often right. A bus that should have come in
> and didn't costs us about **\$4,000** — the road call, the tow, the service we
> lose. Pulling in a bus that was fine costs about **\$250** of a mechanic's
> time. Tell me which one costs me least."*

The two costs are in the setup cell as `COST_MISS` and `COST_PULL`. A *miss* is
a bus below the limit that the model did not flag; a *pull-in* is every bus the
model flagged that was not below the limit. Use the test rows from Part C.

**Store the following values in `results`:**

| key | description |
|---|---|
| `misses_linear` | how many test buses below the limit linear regression failed to flag |
| `misses_knn` | the same for KNN |
| `misses_logistic` | the same for logistic regression |
| `cost_linear` | total cost on the test buses, in dollars, of linear regression's misses and pull-ins |
| `cost_knn` | the same for KNN |
| `cost_logistic` | the same for logistic regression |
| `best_default_model` | the cheapest of the three — one of `"linear_regression"`, `"knn"`, `"logistic_regression"` |

In [ ]:
# YOUR WORKING HERE
pass


In [ ]:
assert "misses_linear" in results, "results['misses_linear'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('lin_model')
_v = _fnfp((_lin() < 3.0).astype(int))[0]
assert _canon("misses_linear", results["misses_linear"], "int") == _canon("misses_linear", _v, "int"), \
    "results['misses_linear'] is not what your stored objects give — report the value your code computes"
assert _fp("misses_linear", _canon("misses_linear", results["misses_linear"], "int")) in {'5ae4780945024917'}, \
    "results['misses_linear'] is not correct yet"
print("ok")

In [ ]:
assert "misses_knn" in results, "results['misses_knn'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('knn_model')
_v = _fnfp(_cls("knn_model"))[0]
assert _canon("misses_knn", results["misses_knn"], "int") == _canon("misses_knn", _v, "int"), \
    "results['misses_knn'] is not what your stored objects give — report the value your code computes"
assert _fp("misses_knn", _canon("misses_knn", results["misses_knn"], "int")) in {'efc5d12d52382fc7'}, \
    "results['misses_knn'] is not correct yet"
print("ok")

In [ ]:
assert "misses_logistic" in results, "results['misses_logistic'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('log_model')
_v = _fnfp(_cls("log_model"))[0]
assert _canon("misses_logistic", results["misses_logistic"], "int") == _canon("misses_logistic", _v, "int"), \
    "results['misses_logistic'] is not what your stored objects give — report the value your code computes"
assert _fp("misses_logistic", _canon("misses_logistic", results["misses_logistic"], "int")) in {'3597492fbaaedf11'}, \
    "results['misses_logistic'] is not correct yet"
print("ok")

In [ ]:
assert "cost_linear" in results, "results['cost_linear'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('lin_model')
_v = _cost((_lin() < 3.0).astype(int))
assert _canon("cost_linear", results["cost_linear"], "int") == _canon("cost_linear", _v, "int"), \
    "results['cost_linear'] is not what your stored objects give — report the value your code computes"
assert _fp("cost_linear", _canon("cost_linear", results["cost_linear"], "int")) in {'58d8555efc937556'}, \
    "results['cost_linear'] is not correct yet"
print("ok")

In [ ]:
assert "cost_knn" in results, "results['cost_knn'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('knn_model')
_v = _cost(_cls("knn_model"))
assert _canon("cost_knn", results["cost_knn"], "int") == _canon("cost_knn", _v, "int"), \
    "results['cost_knn'] is not what your stored objects give — report the value your code computes"
assert _fp("cost_knn", _canon("cost_knn", results["cost_knn"], "int")) in {'d72da789736684ea'}, \
    "results['cost_knn'] is not correct yet"
print("ok")

In [ ]:
assert "cost_logistic" in results, "results['cost_logistic'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('scaler')
_need('log_model')
_v = _cost(_cls("log_model"))
assert _canon("cost_logistic", results["cost_logistic"], "int") == _canon("cost_logistic", _v, "int"), \
    "results['cost_logistic'] is not what your stored objects give — report the value your code computes"
assert _fp("cost_logistic", _canon("cost_logistic", results["cost_logistic"], "int")) in {'02d3692cbb8b7298'}, \
    "results['cost_logistic'] is not correct yet"
print("ok")

In [ ]:
assert "best_default_model" in results, "results['best_default_model'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("best_default_model", _canon("best_default_model", results["best_default_model"], "str")) in {'c3bd8d84a4c0c6be'}, \
    "results['best_default_model'] is not correct yet — use the exact string shown in the brief"
print("ok")

### Part E — Move the threshold, and say what it costs · 20 marks

At its default boundary, scikit-learn's `.predict()` gives class 1 when the
probability is above 0.5, and class 0 at an exact tie (Chapter 8, section 8.3).
Nothing about brake pads makes 0.5 the right line (section 8.6). With a miss costing sixteen times a pull-in, the
manager would rather flag more buses.

In this part, apply our own explicit rule: work with the probability that each
**test** bus is below the limit, and flag a bus when that probability is
**greater than or equal to** the threshold. Try every threshold in `THRESHOLDS`
(in the setup cell).

**Store the following values in `results`:**

| key | description |
|---|---|
| `p_below` | the probabilities, one per test bus, that the bus is below the limit — from your `log_model`, in the same order as `X_test` |
| `cost_by_threshold` | a dict with exactly one entry per threshold in `THRESHOLDS`: threshold → total cost on the test buses, in whole dollars |
| `best_threshold` | the threshold in `THRESHOLDS` with the lowest cost — one of the values in `THRESHOLDS` exactly |
| `misses_at_best` | misses at `best_threshold` |
| `pullins_at_best` | pull-ins at `best_threshold` |
| `extra_pullins` | how many **more** pull-ins `best_threshold` makes than 0.5 does |
| `misses_avoided` | how many **fewer** misses `best_threshold` makes than 0.5 does |
| `saving_vs_default` | cost at 0.5 minus cost at `best_threshold`, in dollars |

The last three are the trade, in the manager's terms: what she gives up, what
she gets, and what it is worth.

In [ ]:
# YOUR WORKING HERE
pass


In [ ]:
assert "p_below" in results, "results['p_below'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need("log_model", "scaler")
_p = _pb()
_m = _model("log_model")
_want = _m.predict_proba(_input(_m))[:, list(_m.classes_).index(1)]
assert np.allclose(_p, _want), "p_below does not match your log_model's probability of class 1 on the test buses, in the same order as X_test — check the column and the scaling"
print("ok")

In [ ]:
assert "cost_by_threshold" in results, "results['cost_by_threshold'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need("p_below")
_c = results["cost_by_threshold"]
assert isinstance(_c, dict), "cost_by_threshold must be a dict"
_norm = {}
for _k, _val in _c.items():
    _g = _grid("cost_by_threshold key", _k)
    assert _g not in _norm, f"cost_by_threshold has two keys for the threshold {_g}"
    _norm[_g] = _int(f"cost_by_threshold[{_g}]", _val)
assert set(_norm) == set(_GRID), "cost_by_threshold needs exactly one entry for each threshold in THRESHOLDS"
_pb_ = _pb()
for _t in _GRID:
    assert _norm[_t] == _cost((_pb_ >= _t).astype(int)), \
        f"the cost at threshold {_t} is not what your p_below gives (flag when probability >= threshold)"
assert _fp("cost_by_threshold", "|".join(str(_norm[t]) for t in _GRID)) == '0b8934867d6b822e', \
    "results['cost_by_threshold'] is not correct yet"
print("ok")

In [ ]:
assert "best_threshold" in results, "results['best_threshold'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_pb_ = _pb()
_v = min(_GRID, key=lambda t: _cost((_pb_ >= t).astype(int)))
assert _canon("best_threshold", results["best_threshold"], "grid") == _canon("best_threshold", _v, "grid"), \
    "results['best_threshold'] is not what your stored objects give — report the value your code computes"
assert _fp("best_threshold", _canon("best_threshold", results["best_threshold"], "grid")) in {'6ca2d4c0029bb055'}, \
    "results['best_threshold'] is not correct yet"
print("ok")

In [ ]:
assert "misses_at_best" in results, "results['misses_at_best'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_need('best_threshold')
_pb_ = _pb()
_cnt = {t: _fnfp((_pb_ >= t).astype(int)) for t in _GRID}
_b = _grid("best_threshold", results["best_threshold"])
_v = _cnt[_b][0]
assert _canon("misses_at_best", results["misses_at_best"], "int") == _canon("misses_at_best", _v, "int"), \
    "results['misses_at_best'] is not what your stored objects give — report the value your code computes"
assert _fp("misses_at_best", _canon("misses_at_best", results["misses_at_best"], "int")) in {'e4ec687c2702dad6'}, \
    "results['misses_at_best'] is not correct yet"
print("ok")

In [ ]:
assert "pullins_at_best" in results, "results['pullins_at_best'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_need('best_threshold')
_pb_ = _pb()
_cnt = {t: _fnfp((_pb_ >= t).astype(int)) for t in _GRID}
_b = _grid("best_threshold", results["best_threshold"])
_v = _cnt[_b][1]
assert _canon("pullins_at_best", results["pullins_at_best"], "int") == _canon("pullins_at_best", _v, "int"), \
    "results['pullins_at_best'] is not what your stored objects give — report the value your code computes"
assert _fp("pullins_at_best", _canon("pullins_at_best", results["pullins_at_best"], "int")) in {'776cafa736477bba'}, \
    "results['pullins_at_best'] is not correct yet"
print("ok")

In [ ]:
assert "extra_pullins" in results, "results['extra_pullins'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_need('best_threshold')
_pb_ = _pb()
_cnt = {t: _fnfp((_pb_ >= t).astype(int)) for t in _GRID}
_b = _grid("best_threshold", results["best_threshold"])
_v = _cnt[_b][1] - _cnt[0.5][1]
assert _canon("extra_pullins", results["extra_pullins"], "int") == _canon("extra_pullins", _v, "int"), \
    "results['extra_pullins'] is not what your stored objects give — report the value your code computes"
assert _fp("extra_pullins", _canon("extra_pullins", results["extra_pullins"], "int")) in {'ce3ffb4e2089e4be'}, \
    "results['extra_pullins'] is not correct yet"
print("ok")

In [ ]:
assert "misses_avoided" in results, "results['misses_avoided'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_need('best_threshold')
_pb_ = _pb()
_cnt = {t: _fnfp((_pb_ >= t).astype(int)) for t in _GRID}
_b = _grid("best_threshold", results["best_threshold"])
_v = _cnt[0.5][0] - _cnt[_b][0]
assert _canon("misses_avoided", results["misses_avoided"], "int") == _canon("misses_avoided", _v, "int"), \
    "results['misses_avoided'] is not what your stored objects give — report the value your code computes"
assert _fp("misses_avoided", _canon("misses_avoided", results["misses_avoided"], "int")) in {'b6d2738034b85bcd'}, \
    "results['misses_avoided'] is not correct yet"
print("ok")

In [ ]:
assert "saving_vs_default" in results, "results['saving_vs_default'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_need('best_threshold')
_pb_ = _pb()
_cnt = {t: _fnfp((_pb_ >= t).astype(int)) for t in _GRID}
_b = _grid("best_threshold", results["best_threshold"])
_v = (_cnt[0.5][0] * 4000 + _cnt[0.5][1] * 250) - (_cnt[_b][0] * 4000 + _cnt[_b][1] * 250)
assert _canon("saving_vs_default", results["saving_vs_default"], "int") == _canon("saving_vs_default", _v, "int"), \
    "results['saving_vs_default'] is not what your stored objects give — report the value your code computes"
assert _fp("saving_vs_default", _canon("saving_vs_default", results["saving_vs_default"], "int")) in {'304ac7f4a156e380'}, \
    "results['saving_vs_default'] is not correct yet"
print("ok")

### Part F — What you would tell the manager · 10 marks

Each entry follows from values you have already calculated, apart from one
short recalculation. Where a value is a **choice**, use the exact string shown.

**Store the following values in `results`:**

| key | description |
|---|---|
| `accuracy_fell` | `True` or `False`: is logistic regression's test accuracy at `best_threshold` **lower** than at 0.5? |
| `zero_misses_guaranteed` | `True` or `False`: does your result on these 240 test buses guarantee that, at `best_threshold`, the fleet will see no misses next year? |
| `best_threshold_if_miss_cost_halved` | redo the Part E search with a miss costing **\$2,000** instead of \$4,000 (pull-ins still \$250). Which threshold in `THRESHOLDS` is cheapest then? |
| `recommendation` | Using only the error costs you calculated for these 240 test inspections, which evaluated model and threshold is the lowest-cost candidate for further evaluation? One of `"linear_regression_at_3mm"`, `"knn"`, `"logistic_at_default"`, `"logistic_at_best_threshold"` |

**On `best_threshold_if_miss_cost_halved`:** the costs are the manager's
estimates. If your recommended threshold changed the moment an estimate moved,
she would need to know.

**On `recommendation`:** this compares the models you evaluated, on one set of
240 inspections. It does not establish that replacing the current schedule will
be safe, or that it will save this amount next year.

In [ ]:
# YOUR WORKING HERE
pass


In [ ]:
assert "accuracy_fell" in results, "results['accuracy_fell'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_need('best_threshold')
_pb_ = _pb()
_cnt = {t: _fnfp((_pb_ >= t).astype(int)) for t in _GRID}
_b = _grid("best_threshold", results["best_threshold"])
_acc = lambda t: float((((_pb_ >= t).astype(int)) == _yte).mean())
_v = _acc(_b) < _acc(0.5)
assert _canon("accuracy_fell", results["accuracy_fell"], "bool") == _canon("accuracy_fell", _v, "bool"), \
    "results['accuracy_fell'] is not what your stored objects give — report the value your code computes"
assert _fp("accuracy_fell", _canon("accuracy_fell", results["accuracy_fell"], "bool")) in {'8a7070e801de4371'}, \
    "results['accuracy_fell'] is not correct yet"
print("ok")

In [ ]:
assert "zero_misses_guaranteed" in results, "results['zero_misses_guaranteed'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("zero_misses_guaranteed", _canon("zero_misses_guaranteed", results["zero_misses_guaranteed"], "bool")) in {'805a1e619551849b'}, \
    "results['zero_misses_guaranteed'] is not correct yet"
print("ok")

In [ ]:
assert "best_threshold_if_miss_cost_halved" in results, "results['best_threshold_if_miss_cost_halved'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
from sklearn.model_selection import train_test_split as _tts
_d = pd.read_csv("data/inspections.csv")
_F = ['km_since_service', 'route_hilliness', 'stops_per_km', 'avg_load', 'braking_score', 'bus_age_years']
_Xtr, _Xte, _mtr, _mte = _tts(_d[_F], _d["pad_mm"], test_size=0.2, random_state=42)
_ytr, _yte = (_mtr < 3.0).astype(int).values, (_mte < 3.0).astype(int).values
def _need(*keys):
    for k in keys:
        assert k in results, f"store results['{k}'] first — this value is checked against it"
def _cols(m):
    # explain: the column order the object was fitted with; arrays carry none, so FEATURES order
    if hasattr(m, "feature_names_in_"):
        c = list(m.feature_names_in_)
        assert sorted(c) == sorted(_F), "a model or the scaler was fitted on columns other than the six FEATURES"
        return c
    return _F
_KIND = {"lin_model": ("LinearRegression", "coef_"), "scaler": ("StandardScaler", "mean_"),
         "knn_model": ("KNeighborsClassifier", "classes_"), "log_model": ("LogisticRegression", "classes_")}
def _model(name):
    # explain: present, the right kind of object, and fitted — checked before anything is asked of it
    _need(name)
    kind, fitted = _KIND[name]
    m = results[name]
    assert type(m).__name__ == kind, f"results['{name}'] must be a fitted {kind}, got {type(m).__name__}"
    assert hasattr(m, fitted), f"{name} is not fitted yet"
    return m
def _feed(m, X):
    # explain: give an object what it was fitted on: named columns in its own order, or a plain array in FEATURES order
    return X[_cols(m)] if hasattr(m, "feature_names_in_") else X[_F].to_numpy()
def _one(p, what):
    # explain: one value per test row; a single (240, 1) column is the same answer, more columns are not
    p = np.asarray(p, dtype=float)
    if p.ndim == 2 and p.shape[1] == 1:
        p = p[:, 0]
    assert p.shape == (240,), f"{what} must give one prediction per test bus (240), got shape {p.shape}"
    return p
def _lin():
    m = _model("lin_model")
    return _one(m.predict(_feed(m, _Xte)), "lin_model")
def _r2():
    # explain: R² of lin_model on the test rows, computed here so any valid target shape gives the same number
    p, t = _lin(), _mte.to_numpy(dtype=float)
    return float(1 - ((t - p) ** 2).sum() / ((t - t.mean()) ** 2).sum())
def _b(m):
    b = np.ravel(m.intercept_)
    assert b.size == 1, "lin_model must predict one target (pad_mm), not several"
    return float(b[0])
def _Z():
    s = _model("scaler")
    c = _cols(s)
    return pd.DataFrame(np.asarray(s.transform(_feed(s, _Xte)), dtype=float), columns=c, index=_Xte.index)
def _input(m):
    return _feed(m, _Z())
def _cls(name):
    m = _model(name)
    return np.asarray(m.predict(_input(m))).astype(int)
def _pb():
    p = results["p_below"]
    if isinstance(p, pd.Series):
        assert p.index.is_unique and set(p.index) == set(_Xte.index), \
            "p_below as a Series must be indexed by the test rows (the index of X_test)"
        p = p.reindex(_Xte.index)
    try:
        p = np.asarray(p, dtype=float).ravel()
    except (TypeError, ValueError):
        raise AssertionError("p_below must hold numbers (probabilities between 0 and 1)") from None
    assert np.all(np.isfinite(p)) and np.all((p >= 0) & (p <= 1)), "p_below must hold probabilities between 0 and 1"
    assert p.shape == (240,), f"p_below must hold one probability per test bus (240), has {p.shape[0]}"
    return p
def _fnfp(pred):
    pred = np.asarray(pred).astype(int)
    return int(((pred == 0) & (_yte == 1)).sum()), int(((pred == 1) & (_yte == 0)).sum())
def _cost(pred, miss=4000, pull=250):
    fn, fp = _fnfp(pred)
    return fn * miss + fp * pull
_need('p_below')
_pb_ = _pb()
_v = min(_GRID, key=lambda t: _cost((_pb_ >= t).astype(int), miss=2000))
assert _canon("best_threshold_if_miss_cost_halved", results["best_threshold_if_miss_cost_halved"], "grid") == _canon("best_threshold_if_miss_cost_halved", _v, "grid"), \
    "results['best_threshold_if_miss_cost_halved'] is not what your stored objects give — report the value your code computes"
assert _fp("best_threshold_if_miss_cost_halved", _canon("best_threshold_if_miss_cost_halved", results["best_threshold_if_miss_cost_halved"], "grid")) in {'995871767c08b6fa'}, \
    "results['best_threshold_if_miss_cost_halved'] is not correct yet"
print("ok")

In [ ]:
assert "recommendation" in results, "results['recommendation'] is not set"
import hashlib, math
import numpy as np
import pandas as pd
_GRID = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
def _fp(key, s):
    return hashlib.sha256(f"a2|{key}|{s}".encode()).hexdigest()[:16]
def _num(key, x):
    # explain: a real number: Python or NumPy, not True/False, not NaN or infinity
    assert not isinstance(x, (bool, np.bool_)), f"results['{key}'] must be a number, not True/False"
    assert isinstance(x, (int, float, np.integer, np.floating)), f"results['{key}'] must be a number"
    try:
        x = float(x)
    except OverflowError:
        x = math.inf
    assert math.isfinite(x), f"results['{key}'] must be a finite number"
    return x
def _int(key, x):
    # explain: 1200, 1200.0 and np.float32(1200) are all the whole number 1200; 1200.5 is not
    x = _num(key, x)
    assert x == int(x), f"results['{key}'] must be a whole number"
    return int(x)
def _grid(key, x):
    # explain: a threshold must be one of THRESHOLDS — only floating-point noise is forgiven
    x = _num(key, x)
    m = [t for t in _GRID if abs(x - t) <= 1e-7]
    assert len(m) == 1, f"results['{key}'] must be one of the values in THRESHOLDS"
    return m[0]
def _canon(key, x, kind):
    if kind == "bool":
        assert isinstance(x, (bool, np.bool_)), f"results['{key}'] must be True or False"
        return "b:" + str(bool(x))
    if kind == "str":
        assert isinstance(x, str), f"results['{key}'] must be a string"
        return "s:" + x
    if kind == "int":
        return "i:" + str(_int(key, x))
    if kind == "grid":
        return "g:" + str(_grid(key, x))
    return "f:" + format(round(_num(key, x), 3) + 0.0, ".3f")
assert _fp("recommendation", _canon("recommendation", results["recommendation"], "str")) in {'7fd129a3e248e3f1'}, \
    "results['recommendation'] is not correct yet — use the exact string shown in the brief"
print("ok")

### AI Usage Declaration · 10 marks

AI use is **permitted** on this assignment, with proper referencing. The
college's AI Usage Declaration is filed **here, in this notebook** — not on D2L.
Fill in the `declaration` dict in the cell below; it asks exactly what the
college form asks. The checks confirm the declaration is **complete** (every
field answered, both statements agreed to). They cannot check whether it is
true. You are responsible for its accuracy, and the academic-integrity policy applies.

**If you did not use AI:** set `used_ai` to `False`, give your name and student
number, agree to both statements, and leave the lists empty. That is a complete
declaration and earns the full 10 marks.

**If you used AI:** complete every field. `details` should say *what* AI produced and
*which parts* it touched (for example, "Copilot suggested the threshold loop in Part E;
I rewrote it"). Use the exact strings shown for the list entries; `"Other: …"`
is allowed in each list with your own text after the colon.

| key | description |
|---|---|
| `name` | your full name, as registered |
| `student_id` | your 9-digit student number, as a string |
| `used_ai` | `True` or `False` |
| `tools` | any of `"ChatGPT"`, `"Copilot"`, `"DALL-E"`, `"Other: …"` — empty if `used_ai` is `False` |
| `purposes` | any of `"Idea Generation"`, `"Writing Assistance"`, `"Image Creation"`, `"Coding/Programming"`, `"Language Refinement"`, `"Other: …"` |
| `details` | what AI produced and where you used it |
| `review` | any of `"Edited AI output for accuracy"`, `"Cross-checked AI output with reliable sources"`, `"Made significant changes to AI output"`, `"Other: …"` |
| `quality_check` | how you made sure the AI-assisted work was right |
| `ethical_use` | how you kept the work your own |
| `reflection` | one or two sentences on how AI helped, or did not |
| `accurate_and_truthful` | `True` — *I declare that the information in this AI Usage Declaration is accurate and truthful, and that I have referenced all AI-generated content used.* |
| `own_work` | `True` — *I declare that the work submitted is my own, that AI was used only as a tool to assist me, and that I am responsible for its content and quality.* |

In [ ]:
declaration = {
    "name": "",
    "student_id": "",
    "used_ai": None,
    "tools": [],
    "purposes": [],
    "details": "",
    "review": [],
    "quality_check": "",
    "ethical_use": "",
    "reflection": "",
    "accurate_and_truthful": False,
    "own_work": False,
}
# YOUR WORKING HERE
pass

for k, v in declaration.items():
    print(f"{k:22} {v!r}")

In [ ]:
assert "declaration" in dir(), "the declaration dict is missing — run the cell above"
d = declaration
assert isinstance(d.get("name"), str) and len(d["name"].strip()) >= 3, "declaration: name is empty"
sid = str(d.get("student_id", "")).strip()
assert sid.isdigit() and len(sid) == 9, f"declaration: student_id must be your 9-digit student number, got {sid!r}"
print("ok")

In [ ]:
d = declaration
assert d.get("used_ai") in (True, False), "declaration: used_ai must be True or False"
TOOLS = ["ChatGPT", "Copilot", "DALL-E"]
PURPOSES = ["Idea Generation", "Writing Assistance", "Image Creation", "Coding/Programming", "Language Refinement"]
ok = lambda xs, allowed: isinstance(xs, list) and all(x in allowed or str(x).startswith("Other: ") for x in xs)
assert ok(d.get("tools"), TOOLS), f"declaration: tools must use the exact strings shown, got {d.get('tools')!r}"
assert ok(d.get("purposes"), PURPOSES), f"declaration: purposes must use the exact strings shown, got {d.get('purposes')!r}"
if d["used_ai"]:
    assert d["tools"], "declaration: used_ai is True but no tool is named"
    assert d["purposes"], "declaration: used_ai is True but no purpose is selected"
else:
    assert not d["tools"] and not d["purposes"], "declaration: used_ai is False but tools/purposes are not empty"
print("ok")

In [ ]:
d = declaration
assert d.get("used_ai") in (True, False), "declaration: used_ai must be True or False"
REVIEW = ["Edited AI output for accuracy", "Cross-checked AI output with reliable sources", "Made significant changes to AI output"]
assert isinstance(d.get("review"), list) and all(x in REVIEW or str(x).startswith("Other: ") for x in d["review"]), \
    f"declaration: review must use the exact strings shown, got {d.get('review')!r}"
for k in ["details", "quality_check", "ethical_use", "reflection"]:
    assert isinstance(d.get(k), str), f"declaration: {k} must be a string"
if d.get("used_ai"):
    assert len(d["details"].strip()) >= 20, "declaration: details — say what AI produced and which stages it touched"
    assert d["review"], "declaration: used_ai is True but no review action is selected"
    for k in ["quality_check", "ethical_use", "reflection"]:
        assert len(d[k].strip()) >= 10, f"declaration: {k} is empty"
print("ok")

In [ ]:
d = declaration
assert d.get("accurate_and_truthful") is True, "declaration: accurate_and_truthful must be True — the declaration is not signed"
assert d.get("own_work") is True, "declaration: own_work must be True — the original-work declaration is not signed"
print("ok")

---

## Before you submit

1. **Kernel → Restart Kernel and Run All Cells.** Every test must print `ok` from a
   clean start. `Validate` runs exactly these tests — there are no hidden ones.
2. **The AI Usage Declaration is filled in**: name, student number, the AI questions, both
   statements. It is worth 10 marks: missing information loses the marks attached
   to the affected checks, and a blank declaration earns zero. Validate tells you
   which field is missing. Nothing is filed on D2L for this assignment.
3. **Don't copy and paste cells in the assignment notebook.** To try something, add a new cell with **+**. Never change a cell's type or the grey cells. If you pasted a copy of a cell, delete the copy before you submit.
4. **Submit** through *Nbgrader → Assignment List* in Jupyter. You can submit more
   than once before the deadline; the last submission is the one marked.
5. Late work: 20 marks out of 100 are deducted for each started day late, to a minimum of 0; work five or more days late receives 0.
